In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
import datetime as dt
from bs4 import BeautifulSoup as bs
import requests
import json
import time
from tqdm import tqdm
import random
import math

In [ ]:
# 메이커 정보 + 프로젝트 정보 df 불러오기
project_df = pd.read_csv('20221120_Wadiz_corpInfo_AllProjects_dropduplicates_df.csv')
# 프로젝트 디테일 정보 df 불러오기
detail_df = pd.read_csv('(20221122-20221124)Wadiz_AllProjects_detail_df_vF.csv')
# 서포터탭 정보 df 불러오기
supporter_df = pd.read_csv('20221120_Wadiz_AllProjects_Supporters_df_Preprocessing_vF.csv')

In [ ]:
# 전체 행/열 보고 싶을 때
pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

# 1. Cross Sectional Data 전처리

## 1. project_df랑 detail_df 칼럼명 변경 후 합치기

In [ ]:
# project_df maker 정보 관련 칼럼명 변경
project_df.rename(columns = {'mainImageUrl':'maker_MainImageUrl',
                             'logoUrl':'maker_LogoUrl', 
                             'makerName':'maker_Name', 
                             'makerIntroduceTags':'maker_Tags', 
                             'coreMessage':'maker_CoreMessage', 
                             'makerSatisfactionAverage':'maker_SatisfactionAverage', 
                             'makerSatisfactionCount':'maker_SatisfactionCount', 
                             'totalAmountAll':'maker_TotalBackedAmount', 
                             'totalCountAll':'maker_TotalSupporterCnt', 
                             'userId':'maker_UserId',
                             'followCount':'maker_TotalFollowerCnt', 
                             'businessRegNumber':'maker_BusinessRegNumber', 
                             'corpShapeName':'maker_CorpShapeName', 
                             'detailLink':'maker_DetailLink'}, inplace = True)

In [ ]:
# project_df 크라우드펀딩 프로젝트 정보 관련 칼럼명 변경
project_df.rename(columns = {'campaignId':'project_Id',
                             'userId_x':'project_UserId', 
                             'isAllOrNothing':'project_AllOrNothing', 
                             'title':'project_Title', 
                             'whenOpen':'project_OpenDate', 
                             'photoUrl':'project_PhotoUrl', 
                             'nickName':'project_MakerName', 
                             'participationCnt':'project_SupporterCnt', 
                             'totalBackedAmount':'project_TotalBackedAmount', 
                             'achievementRate':'project_AchievementRate',
                             'isSuccess':'project_IsSuccess', 
                             'detailLink_x':'project_DetailLink', 
                             'signatureCnt':'project_FacebookCnt', 
                             'remainingDay':'project_RemainingDay', 
                             'endYn':'project_EndYn', 
                             'targetMessage':'project_TargetMessage', 
                             'coreMessage_x':'project_Summary', 
                             'custValueCode':'project_CategoryCode', 
                             'custValueCodeNm':'project_CategoryName', 
                             'isOpen':'project_IsOpen', 
                             'preOpen':'project_PreOpen', 
                             'partnerName':'project_PartnerName', 
                             'hashTag':'project_Tags', 
                             'oneDayRank':'project_OneDayRank', 
                             'isTodayOpen':'project_IsTodayOpen', 
                             'coreMessage_y':'maker_Summary'}, inplace = True)

# 전체 값이 비어있는 칼럼 1차 제거
project_df = project_df.drop(['userId_y', 'detailLink_y'], axis=1)

# project_Id 칼럼 데이터 타입 변경
project_df = project_df.astype({'project_Id':'int'})

In [ ]:
# detail_df 칼럼명 변경
detail_df.rename(columns = {'campaignId':'project_Id'}, inplace = True)

In [ ]:
# project_df와 detail_df 합치기
wadiz_df = pd.DataFrame()
wadiz_df = pd.merge(project_df,detail_df, how='outer',on='project_Id')
wadiz_df

In [ ]:
# csv로 저장
wadiz_df.to_csv('(CrossSectional)Wadiz_df_v1.0.csv', index=False, encoding='utf-8-sig')

## 2. wadiz_df에서 중복되는 칼럼 drop

In [ ]:
# 메이커, 프로젝트 관련 정보 다 합친 df 불러오기
wadiz_df = pd.read_csv('(CrossSectional)Wadiz_df_v1.0.csv')

In [ ]:
# 중복되는 칼럼 1차 제거 (detail_df 중심)
wadiz_df = wadiz_df.drop(['detail_category', 'detail_title', 'project_SupporterCnt', 'detail_summary', 
                          'detail_achievement_rate', 'detail_total_amount', 'detail_total_supporter', 
                          'project_PreOpen'], axis=1)

In [ ]:
# wadiz_df 칼럼명 변경
wadiz_df.rename(columns = {'detail_period':'project_Period', 
                           'detail_news_count':'project_NewsCnt', 
                           'detail_community_count':'project_CommunityCnt', 
                           'detail_supporter_count':'project_SupporterCnt', 
                           'detail_target_amount':'project_TargetAmount', 
                           'detail_preorder':'project_PreOrder', 
                           'detail_reward_cnt':'project_RewardCnt', 
                           'detail_reward':'project_RewardList', 
                           'detail_avg_reward_price':'project_RewardAvgPrice', 
                           'detail_avg_reward_sold_price':'project_RewardAvgSoldPrice', 
                           'detail_likes':'project_Likes'}, inplace = True)

In [ ]:
# csv로 저장
wadiz_df.to_csv('(CrossSectional)Wadiz_df_v1.1.csv', index=False, encoding='utf-8-sig')

## 3. wadiz_df에서 불필요한 칼럼 drop

In [ ]:
# 메이커, 프로젝트 관련 정보 다 합친 df 불러오기
wadiz_df = pd.read_csv('(CrossSectional)Wadiz_df_v1.1.csv')

In [ ]:
# 칼럼 값이 비어있는지 check
wadiz_df['project_IsTodayOpen'].unique()

In [ ]:
# 비어있거나 불필요한 칼럼 제거
wadiz_df = wadiz_df.drop(['maker_MainImageUrl', 'maker_LogoUrl', 'maker_CoreMessage', 'maker_UserId', 
                          'maker_BusinessRegNumber', 'maker_CorpShapeName', 'maker_DetailLink', 
                          'project_PhotoUrl', 'project_MakerName', 'project_RemainingDay', 'project_EndYn', 
                          'project_TargetMessage', 'project_IsOpen', 'project_OneDayRank', 
                          'project_IsTodayOpen'], axis=1)

In [ ]:
# wadiz_df 칼럼명 변경
wadiz_df.rename(columns = {'org_campaignId':'org_HomeCampaignId', 
                           'corpNo':'maker_Id'}, inplace = True)

In [ ]:
# wadiz_df 칼럼 순서 변경
wadiz_df=wadiz_df[['maker_Id', 'maker_Name', 'maker_Tags', 'maker_Summary','maker_SatisfactionAverage', 
                   'maker_SatisfactionCount', 'maker_TotalBackedAmount', 'maker_TotalSupporterCnt', 'maker_TotalFollowerCnt', 
                   'project_Id', 'project_Title', 'project_CategoryCode', 'project_CategoryName', 'project_Tags', 'project_Summary', 
                   'project_PreOrder', 'project_AllOrNothing', 'project_OpenDate', 'project_Period', 
                   'project_IsSuccess', 'project_TargetAmount', 'project_TotalBackedAmount', 'project_AchievementRate',
                   'project_NewsCnt', 'project_CommunityCnt', 'project_SupporterCnt', 'project_FacebookCnt', 'project_Likes', 
                   'project_RewardList', 'project_RewardCnt', 'project_RewardAvgPrice', 'project_RewardAvgSoldPrice', 
                   'project_UserId', 'project_PartnerName', 'project_DetailLink', 'org_HomeCampaignId']]

In [ ]:
# csv로 저장
wadiz_df.to_csv('(CrossSectional)Wadiz_df_v1.2.csv', index=False, encoding='utf-8-sig')

## 4. wadiz_df 칼럼을 각각 분석 가능한 형태로 변경

In [ ]:
# 메이커, 프로젝트 관련 정보 다 합친 df 불러오기
wadiz_df = pd.read_csv('(CrossSectional)Wadiz_df_v1.2.csv')

In [ ]:
# 칼럼 데이터 타입 변경
wadiz_df = wadiz_df.astype({'maker_Id':'int'})
wadiz_df = wadiz_df.astype({'maker_Id':'str', 'project_Id':'str'})
wadiz_df = wadiz_df.astype({'project_CategoryCode':'int'})
wadiz_df = wadiz_df.astype({'project_CategoryCode':'str'})

In [ ]:
# project_NewsCnt, project_CommunityCnt, project_SupporterCnt, project_PreOrder
# project_RewardList, project_RewardCnt, project_RewardAvgPrice, project_RewardAvgSoldPrice 잘못 긁힌 칼럼 다시 긁어오기

# 제대로 안 긁힌 행들 babo_df로 불러오기 
babo_df = wadiz_df[wadiz_df['project_PreOrder']==-1]

# 다시 긁어오기
project_detail_df = pd.DataFrame()

for i in tqdm(range(16021,16057)):
    
    # 베이스 url 정의
    project_url = babo_df['project_DetailLink'][i]
    #HTTP 요청
    project_resp = requests.get(project_url)
    #요청 결과를 json으로 변환
    project_soup = bs(project_resp.text, "html.parser")
    
    # 새소식 수, 커뮤니티 수, 서포터 수 긁어오기
    try:
        soup_TabList = project_soup.select('ul.tab-list')
    except:
        tmp_news_count = '-1'
        tmp_community_count = '-1'
        tmp_supporter_count = '-1'
    try:
        tmp_news_count = soup_TabList[0].select('li a.tab-link')[3].select('span')[0].text
    except:
        tmp_news_count = '0'
    try:
        tmp_community_count = soup_TabList[0].select('li a.tab-link')[4].select('span')[0].text
    except:
        tmp_community_count = '0'
    try:
        tmp_supporter_count = soup_TabList[0].select('li a.tab-link')[5].select('span')[0].text
    except:
        tmp_supporter_count = '0'
    
    # 컨텐츠 오른쪽 정보 긁어오기 (총 달성률, 총 모금액, 총 서포터 수, 프리오더 여부)
    try:
        soup_Right = project_soup.select('div.reward-body-wrap')[0].select('div.wd-ui-sub-contents')[0].select('div.wd-ui-info-wrap')[0].select('div.wd-ui-sub-opener-info')
        tmp_preorder = 1 if soup_Right[0].select('div.project-state-info')[0].select('div.wd-ui-preorder-container') != [] else 0
    except:
        tmp_preorder = '-1'
    
    # 프로젝트 리워드 정보 긁어오기 ([리워드 가격, 배송비, 판매 개수, 제한 수량] 순서)
    # 리워드 개수, 리워드 리스트, 평균 리워드 가격, 평균 리워드 판매 가격
    try:
        tmp_reward_cnt = len(soup_Right[0].select('div.moveRewards')[0].select('div.wd-ui-gift')[0].select('button'))
        tmp_reward = []
        sum_price = 0
        sum_sold_price = 0
        sum_sold_cnt = 0

        for i in range(tmp_reward_cnt):
            tmp = soup_Right[0].select('div.moveRewards')[0].select('div.wd-ui-gift')[0].select('button')[i].select('div.top-info')
            tmp_price = re.sub(r'[^0-9]', '', tmp[0].select('dl.reward-info > dt')[0].text)
            ttmp_shipping = str(tmp[0].select('ul.data-info > li.shipping > em')[0].text if tmp[0].select('ul.data-info > li.shipping > em') != [] else 0)
            tmp_shipping = re.sub(r'[^0-9]', '', ttmp_shipping)
            tmp_sold_cnt = tmp[0].select('p.reward-soldcount > strong')[0].text
            tmp_prepare_cnt = tmp[0].select('p.reward-qty > strong')[0].text if tmp[0].select('p.reward-qty > strong') != [] else tmp_sold_cnt

            sum_price = sum_price + int(tmp_price)
            sum_sold_price = sum_sold_price + (int(tmp_price)*int(tmp_sold_cnt))
            sum_sold_cnt = sum_sold_cnt + int(tmp_sold_cnt)

            tmp_list = [tmp_price, tmp_shipping, tmp_sold_cnt, tmp_prepare_cnt]
            tmp_reward.append(tmp_list)

        tmp_reward = str(tmp_reward)
        avg_reward_price = (sum_price / tmp_reward_cnt) if tmp_reward_cnt != 0 else 0
        avg_reward_sold_price = (sum_sold_price / sum_sold_cnt) if sum_sold_cnt != 0 else 0
    except:
        tmp_reward_cnt = 0
        tmp_reward = '[]'
        avg_reward_price = -1
        avg_reward_sold_price = -1
    
    tmp_detail_df = pd.DataFrame()
    tmp_detail_df = pd.DataFrame({'project_NewsCnt': [tmp_news_count],
                                  'project_CommunityCnt': [tmp_community_count],
                                  'project_SupporterCnt': [tmp_supporter_count],
                                  'project_PreOrder': [tmp_preorder],
                                  'project_RewardList': [tmp_reward],
                                  'project_RewardCnt': [tmp_reward_cnt],
                                  'project_RewardAvgPrice': [avg_reward_price],
                                  'project_RewardAvgSoldPrice': [avg_reward_sold_price]})
    
    # 합친 df를 maker_projects_df에 추가하기
    project_detail_df = project_detail_df.append(tmp_detail_df, ignore_index = True)
    
    # 랜덤으로 sleep 주기
    time.sleep(random.uniform(0.5, 2))

project_detail_df

In [ ]:
# 원래 df에 새로 긁은 df 값들 대체하기
project_detail_df = project_detail_df.set_index(pd.Index(range(16021,16057)))

for i in tqdm(range(16021,16057)):
    wadiz_df.loc[i, 'project_NewsCnt'] = project_detail_df['project_NewsCnt'][i]
    wadiz_df.loc[i, 'project_CommunityCnt'] = project_detail_df['project_CommunityCnt'][i]
    wadiz_df.loc[i, 'project_SupporterCnt'] = project_detail_df['project_SupporterCnt'][i]
    wadiz_df.loc[i, 'project_PreOrder'] = project_detail_df['project_PreOrder'][i]
    wadiz_df.loc[i, 'project_RewardList'] = project_detail_df['project_RewardList'][i]
    wadiz_df.loc[i, 'project_RewardCnt'] = project_detail_df['project_RewardCnt'][i]
    wadiz_df.loc[i, 'project_RewardAvgPrice'] = project_detail_df['project_RewardAvgPrice'][i]
    wadiz_df.loc[i, 'project_RewardAvgSoldPrice'] = project_detail_df['project_RewardAvgSoldPrice'][i]

In [ ]:
# project_AllOrNothing 칼럼 값 변경
for i in tqdm(range(len(wadiz_df))):
    if wadiz_df['project_AllOrNothing'][i] == True:
        wadiz_df.loc[i, 'project_AllOrNothing'] = 1
    else:
        wadiz_df.loc[i, 'project_AllOrNothing'] = 0

In [ ]:
# 펀딩 오픈일 칼럼을 데이트타임 타입으로 형 변환
wadiz_df['project_OpenDate'] = pd.to_datetime(wadiz_df['project_OpenDate'])
wadiz_df['project_OpenDate'] = wadiz_df['project_OpenDate'].dt.strftime('%Y-%m-%d')   # %Y-%m-%d으로 형식 포맷팅
wadiz_df['project_OpenDate'] = pd.to_datetime(wadiz_df['project_OpenDate'])

In [ ]:
# 펀딩 종료일 칼럼 만들고 형 변환
wadiz_df.insert(18, 'project_ClosedDate', wadiz_df['project_Period'].str.split('-', expand = True)[1])
wadiz_df['project_ClosedDate'] = pd.to_datetime(wadiz_df['project_ClosedDate'])
wadiz_df['project_ClosedDate'] = wadiz_df['project_ClosedDate'].dt.strftime('%Y-%m-%d')   # %Y-%m-%d으로 형식 포맷팅
wadiz_df['project_ClosedDate'] = pd.to_datetime(wadiz_df['project_ClosedDate'])

In [ ]:
# 펀딩 기간 칼럼 만들기
wadiz_df.insert(17, 'project_PeriodDays', 0)

for i in tqdm(range(len(wadiz_df))):
    tmp_days = wadiz_df['project_ClosedDate'][i]-wadiz_df['project_OpenDate'][i]
    tmp_days = tmp_days.days + 1
    wadiz_df.loc[i, 'project_PeriodDays'] = tmp_days

In [ ]:
# project_IsSuccess 칼럼 값 변경
for i in tqdm(range(len(wadiz_df))):
    if wadiz_df['project_IsSuccess'][i] == True:
        wadiz_df.loc[i, 'project_IsSuccess'] = 1
    else:
        wadiz_df.loc[i, 'project_IsSuccess'] = 0

In [ ]:
# project_TargetAmount int로 형 변환
for i in tqdm(range(len(wadiz_df))):
    tmp_target = wadiz_df['project_TargetAmount'][i].replace("원","").replace(",", "").strip()
    wadiz_df.loc[i, 'project_TargetAmount'] = tmp_target
    
wadiz_df = wadiz_df.astype({'project_TargetAmount':'int'})

In [ ]:
# project_RewardAvgPrice 칼럼 반올림하기
wadiz_df['project_RewardAvgPrice']= round(wadiz_df['project_RewardAvgPrice'], 1)
wadiz_df['project_RewardAvgSoldPrice']= round(wadiz_df['project_RewardAvgSoldPrice'], 1)

In [ ]:
# org_HomeCampaignId 칼럼 형 변환
wadiz_df = wadiz_df.astype({'org_HomeCampaignId':'str'})

In [ ]:
# 전처리한 칼럼 최종 확인
wadiz_df

In [ ]:
# project_PartnerYn 칼럼 만들기
wadiz_df.insert(36, 'project_PartnerYn', -1)

# True, False 값으로 변경
wadiz_df['project_PartnerYn'] = wadiz_df['project_PartnerName'].isna()

# project_PartnerYn 칼럼 값 변경
for i in tqdm(range(len(wadiz_df))):
    if wadiz_df['project_PartnerYn'][i] == True:
        wadiz_df.loc[i, 'project_PartnerYn'] = 0
    else:
        wadiz_df.loc[i, 'project_PartnerYn'] = 1

In [ ]:
# csv로 저장
wadiz_df.to_csv('(CrossSectional)Wadiz_df_v2.0.csv', index=False, encoding='utf-8-sig')

In [ ]:
df = pd.read_csv('(CrossSectional)Wadiz_df_v2.0.csv')
df

In [ ]:
len(df['maker_Id'].unique())

## 5. 수상 여부 칼럼으로 추가하기

In [ ]:
# 2021 수상 정보 df 불러오기
awards_df = pd.read_csv('2021_Wadiz_Awards_df.csv')

In [ ]:
# 칼럼 데이터 타입 변경
awards_df = awards_df.astype({'corpId':'str'})

In [ ]:
# 수상한 메이커를 리스트로 만들기
award_maker = list(awards_df['corpId'])

In [ ]:
# 수상한 메이커의 크라우드펀딩 프로젝트에는 1을 부여한 칼럼 생성
wadiz_df['awards_Maker'] = -1

for i in tqdm(range(len(wadiz_df))):
    
    tmp_maker = wadiz_df['maker_Id'][i]
    
    if tmp_maker in award_maker:
        tmp_AM = 1
    else:
        tmp_AM = 0
        
    wadiz_df.loc[i, 'awards_Maker'] = tmp_AM
    
wadiz_df

In [ ]:
# csv로 저장
wadiz_df.to_csv('(CrossSectional)Wadiz_df_v3.0.csv', index=False, encoding='utf-8-sig')

## 6. 2021 와디즈 어워즈 발표 이후에 오픈한 프로젝트에 1 부여하는 더미변수 생성

In [ ]:
# 2021 와디즈 어워즈 발표일 변수 선언
Awards21_Date = pd.to_datetime('2021-12-09')

In [ ]:
# project_OpenDate 칼럼 datetime으로 타입 변환
wadiz_df['project_OpenDate'] = pd.to_datetime(wadiz_df['project_OpenDate'])

In [ ]:
# 2021 와디즈 어워즈 발표일 이후에 오픈한 프로젝트에는 1을 부여한 칼럼 생성
wadiz_df['awards_After'] = -1

for i in tqdm(range(len(wadiz_df))):
    
    tmp_opendate = wadiz_df['project_OpenDate'][i]
    diff = tmp_opendate - Awards21_Date
    diff_days = diff.days
    
    if diff_days >= 0:
        tmp_after = 1
    else:
        tmp_after = 0
        
    wadiz_df.loc[i, 'awards_After'] = tmp_after
    
wadiz_df

In [ ]:
# csv로 저장
wadiz_df.to_csv('(CrossSectional)Wadiz_df_v3.1.csv', index=False, encoding='utf-8-sig')

## 7. 매칭을 위한 칼럼 생성

In [ ]:
# wadiz_df 불러오기
wadiz_df = pd.read_csv('(CrossSectional)Wadiz_df_v3.1.csv')

In [ ]:
# 메이커 총 프로젝트 개수 칼럼 만들기

maker_list = list(wadiz_df['maker_Id'].unique())

wadiz_df3 = pd.DataFrame()

for i in tqdm(range(len(maker_list))):
    
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_length = len(tmp_df)
    tmp_df.insert(7, 'maker_TotalProjectCnt', tmp_length)
    wadiz_df3 = wadiz_df3.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df3
wadiz_df

In [ ]:
# project_TargetAmount가 -1이면 비노출 처리된 프로젝트여서 제거
wadiz_df = wadiz_df[wadiz_df.project_TargetAmount != -1]
wadiz_df = wadiz_df.reset_index()   # 인덱스 재정렬
wadiz_df = wadiz_df.drop('index', axis=1)   # 인덱스 칼럼 삭제
wadiz_df

In [ ]:
wadiz_df.columns

In [ ]:
# 달성률이 0-100까지인 칼럼 만들기 (100 이상은 다 100으로 변경)

# project_PartnerYn 칼럼 만들기
wadiz_df.insert(25, 'project_AchievementRate100', wadiz_df['project_AchievementRate'])

for i in tqdm(range(len(wadiz_df))):
    if wadiz_df['project_AchievementRate100'][i]>=100:
        wadiz_df.loc[i, 'project_AchievementRate100'] = 100

In [ ]:
# 메이커별 달성률 평균 칼럼 만들기

wadiz_df2 = pd.DataFrame()

for i in tqdm(range(len(maker_list))):
    
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_mean = tmp_df['project_AchievementRate100'].mean()
    tmp_df.insert(7, 'maker_MeanAchievementRate100', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 2021 와디즈 어워즈 발표일 변수 선언
Awards21_Date = pd.to_datetime('2021-12-09')

# datetime으로 타입 변환
wadiz_df['project_OpenDate'] = pd.to_datetime(wadiz_df['project_OpenDate'])
wadiz_df['project_ClosedDate'] = pd.to_datetime(wadiz_df['project_ClosedDate'])

In [ ]:
# 메이커별 2021 와디즈 어워즈 발표일 이전 달성률 평균 칼럼 만들기

wadiz_df2 = pd.DataFrame()

for i in tqdm(range(len(maker_list))):
    
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_AchievementRate100'].mean()
    
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
        
    tmp_df.insert(8, 'maker_MeanAchievementRate100Before', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커 평균 목표금액 칼럼
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_mean = tmp_df['project_TargetAmount'].mean()
    tmp_df.insert(9, 'maker_MeanTargetAmount', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2021 와디즈 어워즈 발표일 이전 평균 목표금액 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_TargetAmount'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(10, 'maker_MeanTargetAmountBefore', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커 평균 모금액 칼럼
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_mean = tmp_df['project_TotalBackedAmount'].mean()
    tmp_df.insert(7, 'maker_MeanBackedAmount', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2021 와디즈 어워즈 발표일 이전 평균 모금액 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_TotalBackedAmount'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(8, 'maker_MeanBackedAmountBefore', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 프로젝트 후원자 수 칼럼 생성
wadiz_df.insert(37, 'project_BackerCnt', wadiz_df['project_SupporterCnt']-wadiz_df['project_FacebookCnt'])
wadiz_df

In [ ]:
# 메이커 평균 새소식 수 칼럼
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_mean = tmp_df['project_NewsCnt'].mean()
    tmp_df.insert(16, 'maker_MeanNewsCnt', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2021 와디즈 어워즈 발표일 이전 평균 새소식 수 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_NewsCnt'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(17, 'maker_MeanNewsCntBefore', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커 평균 커뮤니티 수 칼럼
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_mean = tmp_df['project_CommunityCnt'].mean()
    tmp_df.insert(18, 'maker_MeanCommunityCnt', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2021 와디즈 어워즈 발표일 이전 평균 커뮤니티 수 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_CommunityCnt'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(19, 'maker_MeanCommunityCntBefore', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커 평균 서포터 수 칼럼
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_mean = tmp_df['project_SupporterCnt'].mean()
    tmp_df.insert(20, 'maker_MeanSupporterCnt', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2021 와디즈 어워즈 발표일 이전 평균 서포터 수 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_SupporterCnt'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(21, 'maker_MeanSupporterCntBefore', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커 평균 후원자 수 칼럼
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_mean = tmp_df['project_BackerCnt'].mean()
    tmp_df.insert(22, 'maker_MeanBackerCnt', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2021 와디즈 어워즈 발표일 이전 평균 후원자 수 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_BackerCnt'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(23, 'maker_MeanBackerCntBefore', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커 평균 지지서명 수 칼럼
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_mean = tmp_df['project_FacebookCnt'].mean()
    tmp_df.insert(24, 'maker_MeanFacebookCnt', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2021 와디즈 어워즈 발표일 이전 평균 지지서명 수 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_FacebookCnt'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(25, 'maker_MeanFacebookCntBefore', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커 평균 찜 수 칼럼
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_mean = tmp_df['project_Likes'].mean()
    tmp_df.insert(26, 'maker_MeanLikes', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2021 와디즈 어워즈 발표일 이전 평균 찜 수 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_Likes'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(27, 'maker_MeanLikesBefore', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커 평균 리워드 가격 칼럼
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_mean = tmp_df['project_RewardAvgPrice'].mean()
    tmp_df.insert(28, 'maker_MeanRewardPrice', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2021 와디즈 어워즈 발표일 이전 평균 리워드 가격 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_RewardAvgPrice'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(29, 'maker_MeanRewardPriceBefore', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# csv로 저장
wadiz_df.to_csv('(CrossSectional)Wadiz_df_v3.2.csv', index=False, encoding='utf-8-sig')

## 8. 분석에 필요하지 않은 행 제거

In [ ]:
# 2020 와디즈 어워즈 발표일 변수 선언
Awards20_Date = pd.to_datetime('2020-12-23')

In [ ]:
# 2020 와디즈 어워즈 발표일 이후에 오픈한 프로젝트만 살리기
wadiz_df = wadiz_df[wadiz_df['project_OpenDate'] > Awards20_Date]
wadiz_df = wadiz_df.reset_index()   # 인덱스 재정렬
wadiz_df = wadiz_df.drop('index', axis=1)   # 인덱스 칼럼 삭제

In [ ]:
# 2021 와디즈 어워즈 발표일 전후로 칼럼 모두 있는 메이커만 살리기

maker_list = list(wadiz_df['maker_Id'].unique())

wadiz2122_df = pd.DataFrame()

for i in tqdm(range(len(maker_list))):
    
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    
    if list(tmp_df['awards_After'].unique()) == [1, 0]:
        wadiz2122_df = wadiz2122_df.append(tmp_df, ignore_index = True)
    elif list(tmp_df['awards_After'].unique()) == [0, 1]:
        wadiz2122_df = wadiz2122_df.append(tmp_df, ignore_index = True)
    
wadiz2122_df

In [ ]:
wadiz_df = wadiz2122_df

# csv로 저장
wadiz_df.to_csv('(CrossSectional)Wadiz_df_v3.3.csv', index=False, encoding='utf-8-sig')

## 9. 2021년 메이커 특징 칼럼 추가

In [ ]:
# 메이커별 2020 와디즈 어워즈 발표일 ~ 2021 와디즈 어워즈 발표일 총 프로젝트 개수 칼럼 만들기

maker_list = list(wadiz_df['maker_Id'].unique())

wadiz_df3 = pd.DataFrame()

for i in tqdm(range(len(maker_list))):
    
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_length = len(tmp_dfB)
    
    if math.isfinite(tmp_length) == True:
        tmp_length = tmp_length
    else:
        tmp_length = -1
    
    tmp_df.insert(14, 'maker_ProjectCnt21', tmp_length)
    wadiz_df3 = wadiz_df3.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df3
wadiz_df

In [ ]:
# 메이커별 2020 와디즈 어워즈 발표일 ~ 2021 와디즈 어워즈 발표일 달성률 평균 칼럼 만들기

wadiz_df2 = pd.DataFrame()

for i in tqdm(range(len(maker_list))):
    
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_AchievementRate100'].mean()
    
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
        
    tmp_df.insert(11, 'maker_MeanAchievementRate100_21', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2020 와디즈 어워즈 발표일 ~ 2021 와디즈 어워즈 발표일 평균 목표금액 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_TargetAmount'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(14, 'maker_MeanTargetAmount21', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2020 와디즈 어워즈 발표일 ~ 2021 와디즈 어워즈 발표일 평균 모금액 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_TotalBackedAmount'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(9, 'maker_MeanBackedAmount21', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2020 와디즈 어워즈 발표일 ~ 2021 와디즈 어워즈 발표일 평균 새소식 수 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_NewsCnt'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(22, 'maker_MeanNewsCnt21', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2020 와디즈 어워즈 발표일 ~ 2021 와디즈 어워즈 발표일 평균 커뮤니티 수 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_CommunityCnt'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(25, 'maker_MeanCommunityCnt21', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2020 와디즈 어워즈 발표일 ~ 2021 와디즈 어워즈 발표일 평균 서포터 수 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_SupporterCnt'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(28, 'maker_MeanSupporterCnt21', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2020 와디즈 어워즈 발표일 ~ 2021 와디즈 어워즈 발표일 평균 후원자 수 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_BackerCnt'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(31, 'maker_MeanBackerCnt21', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2020 와디즈 어워즈 발표일 ~ 2021 와디즈 어워즈 발표일 평균 지지서명 수 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_FacebookCnt'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(34, 'maker_MeanFacebookCnt21', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2020 와디즈 어워즈 발표일 ~ 2021 와디즈 어워즈 발표일 평균 찜 수 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_Likes'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(37, 'maker_MeanLikes21', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# 메이커별 2020 와디즈 어워즈 발표일 ~ 2021 와디즈 어워즈 발표일 평균 리워드 가격 칼럼 만들기
wadiz_df2 = pd.DataFrame()
for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_dfB = tmp_df[tmp_df['awards_After']==0]
    tmp_mean = tmp_dfB['project_RewardAvgPrice'].mean()
    if math.isfinite(tmp_mean) == True:
        tmp_mean = tmp_mean
    else:
        tmp_mean = -1
    tmp_df.insert(40, 'maker_MeanRewardPrice21', tmp_mean)
    wadiz_df2 = wadiz_df2.append(tmp_df, ignore_index = True)
    
wadiz_df = wadiz_df2
wadiz_df

In [ ]:
# csv로 저장
wadiz_df.to_csv('(CrossSectional)Wadiz_df_v4.0.csv', index=False, encoding='utf-8-sig')

In [ ]:
# 만족도 결측치 평균으로 대체

tmp_SaAvg = 0
tmp_SaCnt = 0
Cnt = 0


for i in tqdm(range(len(maker_list))):
    tmp_df = wadiz_df[wadiz_df['maker_Id']==maker_list[i]]
    tmp_df = tmp_df.reset_index()   # 인덱스 재정렬
    tmp_df = tmp_df.drop('index', axis=1)   # 인덱스 칼럼 삭제
    if math.isfinite(tmp_df['maker_SatisfactionAverage'][0]) == True:
        tmp_SaAvg = tmp_SaAvg + tmp_df['maker_SatisfactionAverage'][0]
        tmp_SaCnt = tmp_SaCnt + tmp_df['maker_SatisfactionCount'][0]
        Cnt = Cnt+1

In [ ]:
MeanSaAvg = tmp_SaAvg / Cnt
MeanSaCnt = tmp_SaCnt / Cnt

MeanSaAvg = round(MeanSaAvg, 1)
MeanSaCnt = round(MeanSaCnt)

wadiz_df.loc[wadiz_df['maker_SatisfactionAverage'] != wadiz_df['maker_SatisfactionAverage'], 'maker_SatisfactionAverage'] = MeanSaAvg
wadiz_df.loc[wadiz_df['maker_SatisfactionCount'] != wadiz_df['maker_SatisfactionCount'], 'maker_SatisfactionCount'] = MeanSaCnt

In [ ]:
# csv로 저장
wadiz_df.to_csv('(CrossSectional)Wadiz_df_v4.1.csv', index=False, encoding='utf-8-sig')

## 10. 프로젝트 기간에 맞춰서 wadiz_df 패널로 df 구성하기

In [ ]:
# 날짜 사이 모든 날짜를 리스트로 반환하는 함수 생성
def date_range(start, end):
    start = datetime.strptime(start, "%Y-%m-%d")
    end = datetime.strptime(end, "%Y-%m-%d")
    dates = [(start + timedelta(days=i)).strftime("%Y-%m-%d") for i in range((end-start).days+1)]
    return dates

In [ ]:
# 패널데이터 구성하기

panel_df = pd.DataFrame()

for i in tqdm(range(len(wadiz_df))):
    
    tmp_project = wadiz_df['project_Id'][i]
    tmp_maker = wadiz_df['maker_Id'][i]
    
    tmp_wadiz = wadiz_df[wadiz_df['project_Id']==tmp_project]
    tmp_wadiz = tmp_wadiz.reset_index()   # 인덱스 재정렬
    tmp_wadiz = tmp_wadiz.drop('index', axis=1)   # 인덱스 칼럼 삭제
    
    open_D = tmp_wadiz['project_OpenDate'][0].strftime("%Y-%m-%d")
    closed_D = tmp_wadiz['project_ClosedDate'][0].strftime("%Y-%m-%d")
    dates = date_range(open_D, closed_D)
    
    date_df = pd.DataFrame()
    date_df['Date'] = dates
    date_df.insert(0, 'maker_Id', tmp_maker)
    date_df.insert(1, 'project_Id', tmp_project)
    
    tmp_final = pd.merge(date_df, wadiz_df, how='left', on=['project_Id', 'maker_Id'])
    panel_df = panel_df.append(tmp_final, ignore_index = True)
    
panel_df

In [ ]:
panel_df['Date'] = pd.to_datetime(panel_df['Date'])
panel_df['project_OpenDate'] = pd.to_datetime(panel_df['project_OpenDate'])
panel_df['project_ClosedDate'] = pd.to_datetime(panel_df['project_ClosedDate'])

In [ ]:
# csv로 저장
panel_df.to_csv('(Panel)Wadiz_df_v2.0.csv', index=False, encoding='utf-8-sig')

## 11. panel_df에 supporter_df 합치기

In [ ]:
# supporter_df 칼럼명 변경
supporter_df.rename(columns = {'campaignId':'project_Id', 
                               'corpNo':'maker_Id'}, inplace = True)

In [ ]:
# supporter_df 칼럼 데이터 타입 변경
supporter_df = supporter_df.astype({'maker_Id':'int', 'project_Id':'int'})
supporter_df = supporter_df.astype({'maker_Id':'str', 'project_Id':'str'})

In [ ]:
# 데이트타임으로 타입 변경
supporter_df['Date'] = pd.to_datetime(supporter_df['Date'])

In [ ]:
# panel_df 칼럼 데이터 타입 변경
panel_df = panel_df.astype({'maker_Id':'str', 'project_Id':'str'})

In [ ]:
# panel_df랑 supporter_df 합치기
final_df = pd.merge(panel_df, supporter_df, how='left', on=['project_Id', 'Date', 'maker_Id'])
final_df

In [ ]:
# 결측치 처리
final_df['totalBackerCnt'] = final_df['totalBackerCnt'].fillna(0)
final_df['totalFacebookCnt'] = final_df['totalFacebookCnt'].fillna(0)
final_df['totalSupporterCnt'] = final_df['totalSupporterCnt'].fillna(0)
final_df['visibleBackedAmount'] = final_df['visibleBackedAmount'].fillna(0)
final_df['predictedBackedAmount'] = final_df['predictedBackedAmount'].fillna(0)
final_df['hiddenBackerCnt'] = final_df['hiddenBackerCnt'].fillna(0)
final_df['predictedAdditionalBackedAmount'] = final_df['predictedAdditionalBackedAmount'].fillna(0)

In [ ]:
# 칼럼명 변경
final_df.rename(columns = {'totalBackerCnt':'DailyBackerCnt', 
                           'totalFacebookCnt':'DailyFacebookCnt', 
                           'totalSupporterCnt':'DailySupporterCnt', 
                           'visibleBackedAmount':'DailyVisibleBackedAmount', 
                           'predictedBackedAmount':'DailyPredictedBackedAmount', 
                           'hiddenBackerCnt':'DailyHiddenBackerCnt', 
                           'predictedAdditionalBackedAmount':'DailyPredictedAdditionalBackedAmount'}, inplace = True)

In [ ]:
# avgHiddenBackedAmount, sumVisibleBackedAmount, sumHiddenBackedAmount 결측치 처리

project_list = list(final_df['project_Id'].unique())

df = pd.DataFrame()

for i in tqdm(range(len(project_list))):
    
    tmp_pro = final_df[final_df['project_Id']==project_list[i]]
    tmp_pro = tmp_pro.reset_index()   # 인덱스 재정렬
    tmp_pro = tmp_pro.drop('index', axis=1)   # 인덱스 칼럼 삭제
    
    # 결측치 없는 행만 추출
    tmp_pro2 = tmp_pro[['avgHiddenBackedAmount', 'sumVisibleBackedAmount', 'sumHiddenBackedAmount']].dropna()
    NA_AvgH = tmp_pro2['avgHiddenBackedAmount'].mean()
    NA_SumV = tmp_pro2['sumVisibleBackedAmount'].mean()
    NA_SumH = tmp_pro2['sumHiddenBackedAmount'].mean()
    # 결측치 대체
    tmp_pro.loc[tmp_pro['avgHiddenBackedAmount'] != tmp_pro['avgHiddenBackedAmount'], 'avgHiddenBackedAmount'] = NA_AvgH
    tmp_pro.loc[tmp_pro['sumVisibleBackedAmount'] != tmp_pro['sumVisibleBackedAmount'], 'sumVisibleBackedAmount'] = NA_SumV
    tmp_pro.loc[tmp_pro['sumHiddenBackedAmount'] != tmp_pro['sumHiddenBackedAmount'], 'sumHiddenBackedAmount'] = NA_SumH
    
    df = df.append(tmp_pro, ignore_index = True)
    
df  

In [ ]:
final_df = df

# csv로 저장
final_df.to_csv('(Panel)Wadiz_df_v3.0.csv', index=False, encoding='utf-8-sig')

## 12. initial, middle, ending 칼럼 만들기

In [ ]:
final_df.insert(51, 'project_ElapsedDays', -1)
final_df.insert(52, 'project_ElapsedPercent', -1)

In [ ]:
IME_df = pd.DataFrame()
tmp_list = list(final_df['project_Id'].unique())

for i in range(len(tmp_list)):
    
    tmp_projectId = tmp_list[i]
    sample2 = final_df[final_df['project_Id']==tmp_projectId]
    sample2 = sample2.reset_index()   # 인덱스 재정렬
    sample2 = sample2.drop('index', axis=1)   # 인덱스 칼럼 삭제
    
    for j in range(len(sample2)):
        tmp_days = sample2['Date'][j]-sample2['project_OpenDate'][j]
        tmp_days = tmp_days.days + 1
        tmp_percent = (tmp_days/sample2['project_PeriodDays'][j])*100
        tmp_percent = round(tmp_percent, 1)
        sample2.loc[j, 'project_ElapsedDays'] = tmp_days
        sample2.loc[j, 'project_ElapsedPercent'] = tmp_percent
        
    IME_df = IME_df.append(sample2, ignore_index = True)
    
IME_df

In [ ]:
IME_df['project_ElapsedPercent']

In [ ]:
IME_df['Initial_0_25'] = 0
IME_df['Middle_25_75'] = 0
IME_df['Middle_25_50'] = 0
IME_df['Middle_50_75'] = 0
IME_df['Ending_75_100'] = 0
IME_df['Initial_0_33'] = 0
IME_df['Middle_33_67'] = 0
IME_df['Ending_67_100'] = 0

In [ ]:
IME_df.loc[IME_df['project_ElapsedPercent'] <= 25, 'Initial_0_25'] = 1
IME_df.loc[(IME_df['project_ElapsedPercent'] > 25) & (IME_df['project_ElapsedPercent'] <= 75), 'Middle_25_75'] = 1
IME_df.loc[(IME_df['project_ElapsedPercent'] > 25) & (IME_df['project_ElapsedPercent'] <= 50), 'Middle_25_50'] = 1
IME_df.loc[(IME_df['project_ElapsedPercent'] > 50) & (IME_df['project_ElapsedPercent'] <= 75), 'Middle_50_75'] = 1
IME_df.loc[IME_df['project_ElapsedPercent'] > 75, 'Ending_75_100'] = 1
IME_df.loc[IME_df['project_ElapsedPercent'] <= 33.3, 'Initial_0_33'] = 1
IME_df.loc[(IME_df['project_ElapsedPercent'] > 33.3) & (IME_df['project_ElapsedPercent'] <= 66.7), 'Middle_33_67'] = 1
IME_df.loc[IME_df['project_ElapsedPercent'] > 66.7, 'Ending_67_100'] = 1

In [ ]:
# csv로 저장
IME_df.to_csv('(Panel)Wadiz_df_v3.1.csv', index=False, encoding='utf-8-sig')